# Notebook 03: Entrenamiento y Evaluación del Modelo

**Entrada:** `data/processed/dataset.csv` (generado en el Notebook 02)  
**Salida:** `models/model.pkl` (modelo entrenado listo para la app)

## Objetivos
1. Preparar los datos para el entrenamiento
2. Entrenar un clasificador **Random Forest** con validación cruzada estratificada
3. Comparar contra dos baselines: clasificador de mayoría y Logistic Regression
4. Analizar la importancia de features
5. Guardar el modelo final

## Decisiones de diseño
- **`class_weight='balanced'`** para compensar el desbalance 7:1
- **Cross-validation estratificada k=5** para respetar la proporción de clases en cada fold
- **Métricas principales**: AUC-ROC y F1-score (más informativas que accuracy con desbalance)
- **Sin escalado de features**: Random Forest no lo necesita (es invariante a escala)

## 0. Configuración

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    roc_auc_score, f1_score, classification_report,
    RocCurveDisplay, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from matplotlib.patches import Patch

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR    = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 1. Cargar dataset y preparar X, y

In [ ]:
dataset = pd.read_csv(PROCESSED_DIR / 'dataset.csv')
print(f"Dataset cargado: {dataset.shape[0]:,} filas × {dataset.shape[1]} columnas")
print(f"Distribución de labels: {dataset['label'].value_counts().to_dict()}")

AMINO_ACIDS  = list('ACDEFGHIKLMNPQRSTVWY')
FEATURE_COLS = ['length', 'molecular_weight', 'isoelectric_point', 'gravy'] + \
               [f'aa_{aa}' for aa in AMINO_ACIDS]
META_COLS    = ['protein_id', 'source_molecule', 'source_molecule_iri', 'pathogen']

X = dataset[FEATURE_COLS].values
y = dataset['label'].values

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Features: {FEATURE_COLS}")

## 2. Definir modelos y esquema de validación

Usamos **Stratified K-Fold con k=5**: divide el dataset en 5 partes
manteniendo la proporción de clases en cada fold.
El modelo se entrena 5 veces, cada vez con un fold distinto como test.
Las métricas finales son la media y desviación estándar de los 5 folds.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Mayoría (baseline)': DummyClassifier(
        strategy='most_frequent'
    ),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=42
        ))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
}

print("Modelos definidos:")
for name in models:
    print(f"  - {name}")
print(f"\nValidación: StratifiedKFold k=5, random_state=42")

## 3. Evaluación con cross-validation

Evaluamos los tres modelos con las mismas métricas y el mismo esquema de validación
para que los resultados sean comparables.

In [ ]:
scoring    = ['roc_auc', 'f1', 'f1_macro', 'accuracy']
cv_results = {}

for name, model in models.items():
    print(f"Evaluando: {name}...")
    results = cross_validate(
        model, X, y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )
    cv_results[name] = results
    print(f"  AUC-ROC: {results['test_roc_auc'].mean():.3f} ± {results['test_roc_auc'].std():.3f}")
    print(f"  F1:      {results['test_f1'].mean():.3f} ± {results['test_f1'].std():.3f}")

print("\nEvaluación completada.")

In [ ]:
summary = []
for name, results in cv_results.items():
    summary.append({
        'Modelo':   name,
        'AUC-ROC':  f"{results['test_roc_auc'].mean():.3f} ± {results['test_roc_auc'].std():.3f}",
        'F1':       f"{results['test_f1'].mean():.3f} ± {results['test_f1'].std():.3f}",
        'F1-macro': f"{results['test_f1_macro'].mean():.3f} ± {results['test_f1_macro'].std():.3f}",
        'Accuracy': f"{results['test_accuracy'].mean():.3f} ± {results['test_accuracy'].std():.3f}",
    })

df_summary = pd.DataFrame(summary).set_index('Modelo')
print("Comparativa de modelos (media ± std sobre 5 folds):")
df_summary

## 4. Análisis detallado del Random Forest

Entrenamos el Random Forest sobre todo el dataset para obtener
las curvas ROC, la matriz de confusión y la importancia de features.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(7, 6))
auc_scores = []

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    rf.fit(X_train, y_train)
    y_prob = rf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    auc_scores.append(auc)

    RocCurveDisplay.from_predictions(
        y_test, y_prob,
        name=f"Fold {fold+1} (AUC={auc:.3f})",
        ax=ax, alpha=0.6
    )

ax.plot([0, 1], [0, 1], 'k--', label='Azar (AUC=0.500)')
ax.set_title(
    f'Curvas ROC — Random Forest\nAUC medio: {np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}',
    fontsize=12
)
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(f"AUC por fold: {[f'{a:.3f}' for a in auc_scores]}")
print(f"AUC medio:    {np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}")

In [ ]:
# Matriz de confusión (último fold como ejemplo representativo)
y_pred = rf.predict(X_test)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['No antigénica (0)', 'Antigénica (1)'],
    ax=ax, colorbar=False, cmap='Blues'
)
ax.set_title('Matriz de confusión (fold 5)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print("Classification report (fold 5):")
print(classification_report(y_test, y_pred,
      target_names=['No antigénica (0)', 'Antigénica (1)']))

## 5. Importancia de features

Random Forest calcula automáticamente la importancia de cada feature
basándose en cuánto contribuye cada una a reducir la impureza en los árboles.
Esto nos dice qué características fisicoquímicas son más relevantes
para predecir la antigenicidad.

In [ ]:
rf_full = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_full.fit(X, y)

importances = pd.Series(rf_full.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=False)

top15 = importances.head(15)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2c7bb6' if not f.startswith('aa_') else '#5cb85c' for f in top15.index]
ax.barh(top15.index[::-1], top15.values[::-1], color=colors[::-1])
ax.set_xlabel('Importancia (Gini)')
ax.set_title('Top 15 features por importancia — Random Forest', fontsize=12)
legend_elements = [
    Patch(facecolor='#2c7bb6', label='Fisicoquímica'),
    Patch(facecolor='#5cb85c', label='Composición aminoácido')
]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("Top 15 features:")
for feat, imp in top15.items():
    print(f"  {feat:<25} {imp:.4f}")

fisico_imp = importances[['length', 'molecular_weight', 'isoelectric_point', 'gravy']].sum()
aa_imp     = importances[[f for f in FEATURE_COLS if f.startswith('aa_')]].sum()

print(f"\nImportancia acumulada por grupo:")
print(f"  Features fisicoquímicas (4):  {fisico_imp:.4f} ({fisico_imp:.1%})")
print(f"  Composición aminoácidos (20): {aa_imp:.4f} ({aa_imp:.1%})")

## 6. Guardar el modelo final

Guardamos el modelo entrenado sobre **todo el dataset** (`rf_full`).
Este es el modelo que cargará la app de Streamlit para hacer predicciones.

In [ ]:
MODEL_PATH = MODELS_DIR / 'model.pkl'

model_bundle = {
    'model':        rf_full,
    'feature_cols': FEATURE_COLS,
    'amino_acids':  AMINO_ACIDS,
    'label_map':    {0: 'No antigénica', 1: 'Antigénica'},
    'n_train':      len(X),
    'auc_cv':       float(np.mean(auc_scores)),
}

joblib.dump(model_bundle, MODEL_PATH)
print(f"Modelo guardado en: {MODEL_PATH}")
print(f"Contenido del bundle:")
for k, v in model_bundle.items():
    print(f"  {k}: {type(v).__name__}")

## 7. Resumen final

In [ ]:
rf_auc  = cv_results['Random Forest']['test_roc_auc']
rf_f1   = cv_results['Random Forest']['test_f1']
lr_auc  = cv_results['Logistic Regression']['test_roc_auc']
dum_auc = cv_results['Mayoría (baseline)']['test_roc_auc']

print("=" * 55)
print("RESUMEN DE RESULTADOS")
print("=" * 55)
print(f"  Dataset: {len(X):,} proteínas, {X.shape[1]} features")
print(f"  Validación: StratifiedKFold k=5")
print()
print(f"  {'Modelo':<25} {'AUC-ROC':>10} {'F1':>10}")
print(f"  {'-'*47}")
print(f"  {'Mayoría (baseline)':<25} {dum_auc.mean():>10.3f} {'N/A':>10}")
print(f"  {'Logistic Regression':<25} {lr_auc.mean():>10.3f} {cv_results['Logistic Regression']['test_f1'].mean():>10.3f}")
print(f"  {'Random Forest':<25} {rf_auc.mean():>10.3f} {rf_f1.mean():>10.3f}")
print("=" * 55)
print(f"\n  Mejora RF vs baseline: +{rf_auc.mean() - dum_auc.mean():.3f} AUC-ROC")
print(f"  Modelo guardado en: {MODEL_PATH}")
print(f"  Listo para Notebook 04 (app Streamlit)")